# 01 Dataset Setup

This notebook prepares a reproducible local or Google Colab workflow for loading the MIMIC-III zip archive, extracting it only once, validating the required tables, and writing a setup manifest for downstream notebooks.

Outputs from this notebook are written to `results/manifests/` and the extracted CSV tables live under the configured data directory.

## Why this notebook exists

- Keep dataset access reproducible across local and Colab sessions.
- Avoid hidden notebook state by saving paths and validation outputs.
- Support large MIMIC CSV files with selective loading utilities.
- Prepare a foundation for cohort construction without data leakage shortcuts.

In [11]:
import sys
from pathlib import Path

NOTEBOOK_CWD = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        candidate
        for candidate in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents]
        if (candidate / 'configs' / 'base.yaml').exists() and (candidate / 'src').is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate the project root from the current notebook directory.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.runtime import load_project_runtime

runtime = load_project_runtime(start=PROJECT_ROOT)
IN_COLAB = runtime.in_colab
PROJECT_ROOT = runtime.project_root
config = runtime.config
paths = runtime.paths

PROJECT_ROOT

PosixPath('/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy')

In [12]:
# Optional: install missing dependencies in Colab
if IN_COLAB:
    %pip -q install pyyaml pandas

In [13]:
from src.utils.config import load_config
from src.utils.paths import ensure_directories, resolve_project_paths
from src.utils.logging_utils import write_run_manifest
from src.data_processing.dataset_setup import find_zip_candidates, unzip_dataset
from src.data_processing.io import list_available_tables, validate_required_tables

In [14]:
config

{'project': {'name': 'multimodal-early-sepsis', 'seed': 42, 'timezone': 'UTC'},
 'paths': {'project_root': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy',
  'data_root': 'data',
  'raw_data_dir': 'data/raw',
  'extracted_data_dir': 'data/mimiciii',
  'interim_data_dir': 'results/intermediate',
  'processed_data_dir': 'results/processed',
  'figures_dir': 'figures',
  'tables_dir': 'results/tables',
  'logs_dir': 'results/logs',
  'manifests_dir': 'results/manifests'},
 'dataset': {'default_local_zip_path': 'mimic.zip',
  'zip_filename_patterns': ['mimic', 'MIMIC'],
  'required_tables': ['PATIENTS.csv',
   'ADMISSIONS.csv',
   'ICUSTAYS.csv',
   'CHARTEVENTS.csv',
   'LABEVENTS.csv',
   'NOTEEVENTS.csv',
   'PRESCRIPTIONS.csv',
   'MICROBIOLOGYEVENTS.csv'],
  'optional_tables': ['INPUTEVENTS_MV.csv',
   'INPUTEVENTS_CV.csv',
   'OUTPUTEVENTS.csv',
   'PROCEDUREEVENTS_MV.csv',
   'D_ITEMS.csv',
   'D_LABITEMS.csv'],
  'chunksize': 100000,
  'low_memory': True},
 'colab': {'d

## Locate the zip archive

For local runs, this notebook now prefers `mimic.zip` in the project root. In Colab, it searches your Drive root. If there are multiple zip files, pick the correct one explicitly.

In [15]:
default_local_zip_path = paths['project_root'] / config['dataset']['default_local_zip_path']
search_root = Path(config['colab']['default_drive_search_root']) if IN_COLAB else default_local_zip_path.parent

zip_candidates = []
if not IN_COLAB and default_local_zip_path.exists():
    zip_candidates.append(default_local_zip_path)

zip_candidates.extend(
    path
    for path in find_zip_candidates(
        search_root=search_root,
        filename_patterns=config['dataset']['zip_filename_patterns'],
    )
    if path not in zip_candidates
)
zip_candidates[:20]

[PosixPath('/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy/mimic.zip')]

In [16]:
# For local runs, this defaults to PROJECT_ROOT / 'mimic.zip' when present.
ZIP_PATH = default_local_zip_path if default_local_zip_path.exists() else (zip_candidates[0] if zip_candidates else None)
ZIP_PATH

PosixPath('/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy/mimic.zip')

In [17]:
assert ZIP_PATH is not None, 'No MIMIC zip archive found. Set ZIP_PATH manually.'
extracted_members = unzip_dataset(
    zip_path=ZIP_PATH,
    destination_dir=paths['extracted_data_dir'],
    overwrite=False,
)
len(extracted_members)

31

In [18]:
available_tables = list_available_tables(paths['extracted_data_dir'])
required_status = validate_required_tables(
    paths['extracted_data_dir'],
    config['dataset']['required_tables'],
)
available_tables[:20], required_status

(['ADMISSIONS.csv',
  'CALLOUT.csv',
  'CAREGIVERS.csv',
  'CHARTEVENTS.csv',
  'CPTEVENTS.csv',
  'DATETIMEEVENTS.csv',
  'DIAGNOSES_ICD.csv',
  'DRGCODES.csv',
  'D_CPT.csv',
  'D_ICD_DIAGNOSES.csv',
  'D_ICD_PROCEDURES.csv',
  'D_ITEMS.csv',
  'D_LABITEMS.csv',
  'ICUSTAYS.csv',
  'INPUTEVENTS_CV.csv',
  'INPUTEVENTS_MV.csv',
  'LABEVENTS.csv',
  'MICROBIOLOGYEVENTS.csv',
  'NOTEEVENTS.csv',
  'OUTPUTEVENTS.csv'],
 {'PATIENTS.csv': True,
  'ADMISSIONS.csv': True,
  'ICUSTAYS.csv': True,
  'CHARTEVENTS.csv': True,
  'LABEVENTS.csv': True,
  'NOTEEVENTS.csv': True,
  'PRESCRIPTIONS.csv': True,
  'MICROBIOLOGYEVENTS.csv': True})

In [19]:
manifest_path = paths['manifests_dir'] / '01_dataset_setup_manifest.json'
write_run_manifest(
    path=manifest_path,
    stage='01_dataset_setup',
    config=config,
    extra={
        'zip_path': str(ZIP_PATH),
        'default_local_zip_path': str(default_local_zip_path),
        'search_root': str(search_root),
        'extracted_dir': str(paths['extracted_data_dir']),
        'available_tables': available_tables,
        'required_status': required_status,
    },
)
manifest_path

PosixPath('/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy/results/manifests/01_dataset_setup_manifest.json')

## Sanity-check one table with selective loading

This final check shows how later notebooks will load only the necessary columns instead of reading entire MIMIC tables into memory.

In [20]:
from src.data_processing.io import load_table

patients_preview = load_table(
    extracted_dir=paths['extracted_data_dir'],
    table_name='PATIENTS.csv',
    usecols=['SUBJECT_ID', 'GENDER', 'DOB'],
    nrows=5,
    low_memory=config['dataset']['low_memory'],
)
patients_preview

,SUBJECT_ID,GENDER,DOB
0,249,F,2075-03-13 00:00:00
1,250,F,2164-12-27 00:00:00
2,251,M,2090-03-15 00:00:00
3,252,M,2078-03-06 00:00:00
4,253,F,2089-11-26 00:00:00


## Next notebook

`02_data_exploration.ipynb` will profile table schemas, cohort counts, note coverage, and missingness patterns needed for rigorous cohort construction.